# 04 — Carga no Postgres

**Objetivo:** criar o schema `caged` (dims + fato), carregar dicionários e a camada curada, e rodar **queries de sanidade**.

> Pré-requisito: `docker compose up -d db` ativo.

In [ ]:
import sys, pathlib
# garante que o pacote caged_etl (em ../) seja importável a partir do notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from caged_etl import setup_logging, get_settings
setup_logging()
settings = get_settings()
print('Municípios:', settings.municipios, '| Anos:', settings.ano_inicio, '-', settings.ano_fim)

## 1. Criar schema + carregar dims e fato (recarga limpa)

In [ ]:
from caged_etl import load
n = load.run_load(recreate=True)
print('linhas carregadas no fato:', n)

## 2. Sanidade — contagem por município (deve ser só 270430 e 270030)

In [ ]:
import polars as pl
from caged_etl.schema import get_engine, SCHEMA
eng = get_engine()
q = f'''SELECT f.municipio, m.descricao AS municipio_nome, count(*) AS n
        FROM {SCHEMA}.fato_caged_mov f
        JOIN {SCHEMA}.dim_municipio m ON m.codigo = f.municipio
        GROUP BY 1,2 ORDER BY n DESC'''
pl.read_database(q, connection=eng)

## 3. Sanidade — saldo de emprego mensal por município
(admissões − desligamentos = soma de `saldo_movimentacao`)

In [ ]:
q = f'''SELECT f.competenciamov, m.descricao AS municipio,
               SUM(f.saldo_movimentacao) AS saldo
        FROM {SCHEMA}.fato_caged_mov f
        JOIN {SCHEMA}.dim_municipio m ON m.codigo = f.municipio
        GROUP BY 1,2 ORDER BY 1,2'''
saldo = pl.read_database(q, connection=eng)
print(saldo.head(24))
saldo

## 4. Sanidade — decode via JOIN (salário médio por sexo)

In [ ]:
q = f'''SELECT m.descricao AS municipio, s.descricao AS sexo,
               ROUND(AVG(f.salario),2) AS salario_medio, count(*) AS n
        FROM {SCHEMA}.fato_caged_mov f
        JOIN {SCHEMA}.dim_municipio m ON m.codigo = f.municipio
        JOIN {SCHEMA}.dim_sexo s ON s.codigo = f.sexo
        GROUP BY 1,2 ORDER BY 1,2'''
pl.read_database(q, connection=eng)

## ✅ ETL concluído
Dados curados disponíveis no Postgres (`schema caged`) para o backend/frontend.